# Nettoyage des données du Congrès (2014-2026) — donnée propre pour backtest

**Objectif.** Partir de notre donnée validée 2014-2026 et produire, *étape par étape*, un jeu de données
**propre et prêt pour un backtest**. Chaque filtre est **justifié** et **chiffré** (entonnoir).

**Principe.**
- **Source** : `common.quality.load_final` — notre panel *golden* 2014-2026 (158 172 transactions uniques),
  déjà dédupliqué et corrigé (dates + tickers) à la lecture. *Quiver n'est jamais réinjecté* ; on n'utilise
  **pas** la table hybride 2014-2026 (qui, elle, contient du Quiver).
- **Nettoyage strict** : à chaque étape on **retire** les lignes non-backtestables (on ne les garde pas avec
  un simple drapeau). Périmètre retenu : **actions cotées + ETF uniquement**.
- **Réutilisation** : tous les filtres s'appuient sur des fonctions **déjà écrites et testées** du dépôt —
  on ne réécrit aucune logique de nettoyage.

**Sortie** : `data/clean/transactions_backtest_2014_2026.csv`.

> **⚠ Anti-look-ahead (backtest).** Le fichier exporte `transaction_date` ET `disclosure_date`. Un backtest doit **entrer sur `disclosure_date`** (date de dépôt imprimée, fiable) : c'est la première date où l'info est publique. Quelques `transaction_date` issues de l'OCR restent imprécises (~2 %), sans impact si l'entrée se fait sur la divulgation.

In [1]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# --- localisation de la racine du dépôt (dossier contenant common/ ET data/) ---
# Le code+données S1/S2 ont été regroupés sous « 00_S1S2_donnees/ ». On teste le cwd et ses parents
# (cas normal : ce notebook est ouvert depuis 00_S1S2_donnees/), puis des candidats explicites au cas où
# Jupyter démarrerait à la RACINE du dépôt — 00_S1S2_donnees/ en est alors un ENFANT, jamais atteint par
# la simple remontée vers les parents.
def _find_repo():
    here = Path.cwd()
    cands = [here, *here.parents,
             here / "00_S1S2_donnees",
             Path.home() / "Downloads" / "Jupiter" / "00_S1S2_donnees",
             Path.home() / "Downloads" / "Jupiter"]            # repli : ancienne disposition (data/ à la racine)
    for c in cands:
        if (c / "common" / "quality.py").exists() and (c / "data" / "house" / "tables").exists():
            return c
    raise RuntimeError("Racine du dépôt (contenant common/ et data/) introuvable")

REPO = _find_repo()
sys.path.insert(0, str(REPO))

# --- fonctions réutilisées (aucune logique de nettoyage réécrite ici) ---
from common.quality import load_final, _asset_bucket    # chargement + famille d'actif
from house.quiver import norm_ticker                     # normalisation du ticker
from common.quiver_diagnosis import _quiver_untradeable  # ticker non coté (CUSIP, $, fragment OCR…)

print("Dépôt :", REPO)

# --- petit utilitaire pour tracer l'entonnoir (étape, filtre, retirées, restantes) ---
funnel = []
def _step(code, label, before, after):
    funnel.append({"étape": code, "filtre": label,
                   "retirées": before - after, "restantes": after})
    print(f"[{code}] {label}\n    {before:,} → {after:,}   (retirées : {before - after:,})")

Dépôt : /Users/lemairealice/Downloads/Jupiter/00_S1S2_donnees


## Chargement — notre donnée validée 2014-2026

`load_final` assemble les 4 sous-corpus (House / Sénat × électronique / OCR), déduplique les re-divulgations
d'une année sur l'autre, applique les corrections *read-time* (dates, tickers) et dérive les colonnes utiles
(`lag_days`, `op`, `amount_midpoint`, `corpus`…). On n'a donc rien à recalculer en amont.

In [2]:
df = load_final(REPO)
N0 = len(df)
print(f"{N0:,} transactions uniques chargées\n")
print("Répartition par sous-corpus :")
print(df["corpus"].value_counts().to_string())

158,172 transactions uniques chargées

Répartition par sous-corpus :
corpus
House OCR             87011
House électronique    54150
Sénat électronique    13026
Sénat OCR              3985


## Normalisation — palier ouvert « > $50M » au plancher (cohérence inter-corpus)

`amount_midpoint` est **relu** des tables golden ; or House OCR chiffrait le palier ouvert « Over $50,000,000 »
à **$75M** (1,5× plancher) alors que les 3 autres corpus utilisent le **plancher $50M**. On **harmonise au
plancher** — convention conservatrice, alignée sur le palier « > $1M » déjà planché partout. *Aucune ligne
retirée : c'est une correction de valeur, l'entonnoir A→D reste inchangé.*

In [3]:
# Palier ouvert « > $50M » : plancher cohérent inter-corpus (House OCR mettait 75M ; les autres, 50M).
palier_ouvert = df["amount_range"].astype(str).str.strip().eq("Over $50,000,000")
n_fix = int((palier_ouvert & (df["amount_midpoint"] != 50_000_000)).sum())
df.loc[palier_ouvert, "amount_midpoint"] = 50_000_000.0
print(f"Palier ouvert « > $50M » harmonisé au plancher : {n_fix} ligne(s) corrigée(s) (75M → 50M).")

Palier ouvert « > $50M » harmonisé au plancher : 2 ligne(s) corrigée(s) (75M → 50M).


## Étape A — Dates présentes et cohérentes

Un backtest a besoin d'une **chronologie fiable**. On retire les lignes où :
- une date (transaction *ou* divulgation) est **illisible / absente** → `lag_days` vaut `NaN` (on ne sait pas
  *quand* agir) ;
- la **divulgation précède la transaction** (`lag_days < 0`) : c'est **impossible** (on « saurait » avant que
  le trade existe) — ce sont des coquilles du déposant ou des erreurs d'OCR ;
- garde-fou : l'**année de transaction** est hors plage plausible (avant 2012 ou après l'année de dépôt).

`lag_days` (= divulgation − transaction, en jours) est **déjà calculé** par `load_final`.

In [4]:
n = len(df)
fy = pd.to_numeric(df["file_year"], errors="coerce")
mask_parse    = df["lag_days"].notna()                        # dates lisibles
mask_coherent = df["lag_days"] >= 0                           # divulgation ≥ transaction
mask_year     = (df["txn_year"] >= 2012) & (df["txn_year"] <= fy)   # année plausible
keep = mask_parse & mask_coherent & mask_year

# aperçu : quelques dates incohérentes retirées (divulgation AVANT transaction)
apercu = (df[mask_parse & (df["lag_days"] < 0)]
          [["declarant_name", "ticker", "transaction_date", "disclosure_date", "lag_days"]].head(5))
print("Exemples de dates incohérentes retirées (divulgation avant transaction) :")
print(apercu.to_string(index=False), "\n")

df = df[keep].copy()
_step("A", "dates présentes & cohérentes", n, len(df))

Exemples de dates incohérentes retirées (divulgation avant transaction) :
  declarant_name ticker transaction_date disclosure_date  lag_days
     Kevin Yoder   SMLP       2014-12-17      2014-12-16      -1.0
Randy Neugebauer    QRE       2014-06-24      2014-06-20      -4.0
Randy Neugebauer    NaN       2014-05-15      2014-05-14      -1.0
  Nick J. Rahall     FB       2014-03-20      2014-03-12      -8.0
Patrick J Toomey    PFS       2014-11-30      2014-11-05     -25.0 

[A] dates présentes & cohérentes
    158,172 → 157,615   (retirées : 557)


## Étape B — Actions cotées et ETF uniquement

Un backtest a besoin d'un **prix**, donc d'un **ticker coté valide**. On garde une ligne si :
- son ticker se **normalise en un symbole non vide** (`norm_ticker`), **et**
- ce ticker est **réellement coté** (on écarte via `_quiver_untradeable` les CUSIP, préférentielles `$`,
  fragments d'OCR, échéances obligataires…), **et**
- sa **famille d'actif n'est pas explicitement non-cotée** (`_asset_bucket` ∉ {obligation, muni, gouvernement,
  option, « autre »}).

Concrètement, un **ticker « valide »** = un symbole qu'on peut relier à un **prix de marché** : **≤ 5 caractères**, lettres/chiffres, rien de bizarre.

| Ticker | Gardé ? | Raison |
|---|---|---|
| `AAPL`, `XOM`, `REGL` | ✅ | vrai symbole → on a un prix |
| *(vide)*, `NaN` | ❌ | aucun symbole |
| `BXS$A` | ❌ | le `$` = action préférentielle |
| `PFE  VTRS` | ❌ | espace = deux tickers collés (ligne d'échange) |
| CUSIP `037833100` | ❌ | numéro comptable, pas un ticker |

**On raisonne « ticker d'abord ».** Une ligne au ticker coté valide est gardée *même si son `asset_type` est
vide* — cas fréquent en House OCR : des milliers de vraies actions (NVDA, IBM, ASML…) sans étiquette de type.
Un filtre « type d'abord » les jetterait à tort (≈ 3 000 lignes).

En pratique, sur les 30 207 retirées : **~91 %** n'ont **aucun ticker** (non valorisables), **12 %** sont des
**options / obligations**, **< 1 %** des tickers malformés. La cellule d'audit ci-dessous le **prouve**,
chiffres à l'appui — rien n'est retiré « en aveugle ».

In [5]:
n = len(df)
NON_COTE = {"bond", "muni", "gov", "option", "autre"}
mask_ticker   = df["ticker"].map(norm_ticker) != ""                    # symbole non vide
mask_tradable = ~df["ticker"].map(_quiver_untradeable)                 # réellement coté
mask_famille  = ~df["asset_type"].map(_asset_bucket).isin(NON_COTE)    # pas une famille non-cotée
keep = mask_ticker & mask_tradable & mask_famille

apercu = df[~keep][["declarant_name", "asset_description", "asset_type", "ticker"]].head(5)
print("Exemples de lignes retirées (non cotées / non tickérisées) :")
print(apercu.to_string(index=False), "\n")

avant_B = df                       # snapshot avant filtrage (réutilisé par la cellule d'audit ci-dessous)
df = df[keep].copy()
_step("B", "actions + ETF cotés (ticker-first)", n, len(df))

Exemples de lignes retirées (non cotées / non tickérisées) :
  declarant_name                  asset_description asset_type ticker
Randy Neugebauer     QR ENERGY, LP 9.25% 08/01/2020        NaN    NaN
Randy Neugebauer     QR ENERGY, LP 9.25% 08/01/2020        NaN    NaN
Randy Neugebauer     QR ENERGY, LP 9.25% 08/01/2020        NaN    NaN
Randy Neugebauer     QR ENERGY, LP 9.25% 08/01/2020        NaN    NaN
    Lois Frankel Merck & Company, Inc. Common stock        NaN    NaN 



[B] actions + ETF cotés (ticker-first)
    157,615 → 127,408   (retirées : 30,207)

### Pourquoi ces 30 207 lignes partent — contrôle chiffré

Pour lever tout doute, on **décompose** les lignes écartées par **cause exclusive** (chaque ligne n'a qu'une
seule raison), on vérifie l'**étanchéité** (aucun non-coté ne doit survivre dans les gardées) et on chiffre
le **rescue « ticker-first »**. *Aucun re-filtrage : cette cellule ne fait qu'expliquer l'étape B.*

In [6]:
# On réutilise `avant_B` (état AVANT filtrage) + les 3 masques calculés à l'étape B. On ne re-filtre RIEN.
fam = avant_B["asset_type"].map(_asset_bucket)
cause = pd.Series(index=avant_B.index, dtype="object")
cause[~mask_ticker]                                = "1. ticker VIDE (aucun symbole → pas de prix)"
cause[mask_ticker & ~mask_tradable]                = "2. ticker MALFORMÉ ($, espace, fragment OCR)"
cause[mask_ticker & mask_tradable & ~mask_famille] = "3. OPTION / OBLIGATION (famille non-cotée)"
ret = cause.dropna()

print(f"Pourquoi les {len(ret):,} lignes de l'étape B partent (une seule cause par ligne) :")
print(ret.value_counts().sort_index().to_string())
print()

print(f"Contrôle d'étanchéité — familles des {int(keep.sum()):,} GARDÉES (equity/ETF seulement) :")
print(fam[keep].value_counts().to_string())
fuite = int(fam[keep].isin(list(NON_COTE)).sum())
print(f"  → non-cotées ayant fui dans les gardées : {fuite}   (0 = filtre étanche)")
print()

rescue = int((keep & (fam == "manquant")).sum())
print(f"Rescue « ticker-first » : {rescue:,} actions/ETF à asset_type vide GARDÉES grâce au ticker.")

Pourquoi les 30,207 lignes de l'étape B partent (une seule cause par ligne) :
1. ticker VIDE (aucun symbole → pas de prix)    27453
2. ticker MALFORMÉ ($, espace, fragment OCR)      163
3. OPTION / OBLIGATION (famille non-cotée)       2591

Contrôle d'étanchéité — familles des 127,408 GARDÉES (equity/ETF seulement) :
asset_type
action      110019
manquant     15769
fonds         1620
  → non-cotées ayant fui dans les gardées : 0   (0 = filtre étanche)

Rescue « ticker-first » : 15,769 actions/ETF à asset_type vide GARDÉES grâce au ticker.


## Étape C — Direction exploitable (achat / vente)

Pour simuler une position, il faut un **sens clair**. On garde les **achats** et les **ventes** ; on retire les
`exchange` et les rares `other`, dont l'économie est ambiguë. *On ne présume pas la stratégie* : conserver les
deux sens laisse le backtest décider (long / short, étude d'événement, sorties). Le sens est déjà normalisé
dans la colonne `op` par `load_final`.

In [7]:
n = len(df)
df = df[df["op"].isin(["buy", "sell"])].copy()
_step("C", "direction ∈ {achat, vente}", n, len(df))

[C] direction ∈ {achat, vente}
    127,408 → 126,671   (retirées : 737)


## Étape D — Montant présent

Pour **dimensionner** une position (en notionnel), il faut un montant. On retire les lignes sans
`amount_midpoint` numérique (le point milieu de la fourchette déclarée, déjà calculé par `load_final`).
Coût négligeable (< 1 %).

In [8]:
n = len(df)
df = df[df["amount_midpoint"].notna()].copy()
_step("D", "montant présent (amount_midpoint)", n, len(df))

[D] montant présent (amount_midpoint)
    126,671 → 125,984   (retirées : 687)


## Récapitulatif de l'entonnoir

In [9]:
rows = [{"étape": "—", "filtre": "départ (load_final)", "retirées": 0, "restantes": N0}] + funnel
recap = pd.DataFrame(rows)
print(recap.to_string(index=False))
print(f"\nDonnée propre finale : {len(df):,} lignes  ({100 * len(df) / N0:.1f} % du panel de départ)\n")

print("Par chambre :");     print(df["chamber"].value_counts().to_string())
print("\nPar sous-corpus :"); print(df["corpus"].value_counts().to_string())
print("\nPar sens :");        print(df["op"].value_counts().to_string())

étape                             filtre  retirées  restantes
    —                départ (load_final)         0     158172
    A       dates présentes & cohérentes       557     157615
    B actions + ETF cotés (ticker-first)     30207     127408
    C         direction ∈ {achat, vente}       737     126671
    D  montant présent (amount_midpoint)       687     125984

Donnée propre finale : 125,984 lignes  (79.7 % du panel de départ)

Par chambre :
chamber
house     114334
senate     11650

Par sous-corpus :
corpus
House OCR             72316
House électronique    42018
Sénat électronique     9729
Sénat OCR              1921

Par sens :
op
buy     64701
sell    61283


## Export — `data/clean/transactions_backtest_2014_2026.csv`

18 colonnes en `snake_case`, prêtes pour le backtest : **identité du membre**, **caractéristiques du trade**,
les **deux dates** (`transaction_date`, `disclosure_date`) et des colonnes de **traçabilité**
(`doc_id`, `provenance`, `natural_key_hash`).

In [10]:
OUT_DIR = REPO / "data" / "clean"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT = OUT_DIR / "transactions_backtest_2014_2026.csv"

COLS = ["bioguide_id", "member_name", "party", "chamber", "state_district", "committee_membership",
        "ticker", "asset_type", "sector_gics", "etf_proxy", "direction", "amount_midpoint",
        "amount_range", "transaction_date", "disclosure_date",
        "doc_id", "provenance", "natural_key_hash"]

export = (df.rename(columns={"declarant_name": "member_name", "op": "direction"})
            .reindex(columns=COLS))
export.to_csv(OUT, index=False)

print(f"Écrit : {OUT}")
print(f"{len(export):,} lignes × {export.shape[1]} colonnes\n")
print(export.head().to_string(index=False))

Écrit : /Users/lemairealice/Downloads/Jupiter/00_S1S2_donnees/data/clean/transactions_backtest_2014_2026.csv
125,984 lignes × 18 colonnes

bioguide_id  member_name      party chamber state_district                              committee_membership ticker asset_type            sector_gics etf_proxy direction  amount_midpoint     amount_range transaction_date disclosure_date   doc_id           provenance                                                 natural_key_hash
    G000563    Bob Gibbs Republican   house           OH07                                               NaN   AAPL        NaN Information Technology       XLK       buy           8000.5 $1,001 - $15,000       2014-12-30      2014-12-30 20002288 house-pdf-electronic 560fd99e7998a4b792946c2abed4dd01aefed3d5396066067c773ac41559bc6a
    G000563    Bob Gibbs Republican   house           OH07                                               NaN   INTC        NaN Information Technology       XLK       buy           8000.5 $1,001 - $